In [1]:
# Requisitos: pandas, numpy, matplotlib, scipy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ---------- 1. carregar dados ----------
# substitua url_csv por caminho local se baixou o CSV
url_csv = 'daily-min-temperatures.csv'  # ex: 'DailyDelhiClimateTrain.csv' ou o arquivo do UCI
df = pd.read_csv(url_csv)       # espera colunas: Date, Temp (ou renomeie abaixo)
# exemplo: ajustar nomes conforme o CSV
# df = df.rename(columns={'DailyMinimumTemperatures':'Temp', 'Date':'date'})
# converter data:
df['date'] = pd.to_datetime(df['Date'])
df = df.sort_values('date').reset_index(drop=True)
y = df['Temp'].astype(float)    # série Y_n

# ---------- 2. construir previsor honesto simples ----------
k = 7   # janela de média móvel (ajuste em aula)
# hatY_n (usamos média móvel que isola previsor que usa somente info até n-1)
hatY = y.shift(1).rolling(window=k, min_periods=1).mean()  # média dos k dias imediatamente anteriores

# ---------- 3. definir erros e processo acumulado ----------
M = y - hatY
S = M.cumsum()

# armazenar no dataframe para análise
df2 = pd.DataFrame({'date': df['date'], 'Y': y, 'hatY': hatY, 'M': M, 'S': S})

# ---------- 4. verificação empírica: média condicional aproximada ----------
# abordagem prática: agrupar por quantis do informação passada (p.ex. valor de Y_{n-1} ou média passada)
df2['lag1'] = df2['Y'].shift(1)
# dividir em bins do lag1 e calcular média de M em cada bin
df2_drop = df2.dropna(subset=['M','lag1'])
bins = pd.qcut(df2_drop['lag1'], q=5, duplicates='drop')
cond_means = df2_drop.groupby(bins)['M'].mean()
cond_counts = df2_drop.groupby(bins)['M'].count()

print("Média de M condicional por bins de Y_{n-1}:")
print(pd.concat([cond_means, cond_counts], axis=1).rename(columns={'M':'mean_M','count':'n'}))

# teste global: média de M diferente de zero?
mean_M = df2_drop['M'].mean()
tstat, pval = stats.ttest_1samp(df2_drop['M'].dropna(), 0.0)
print(f"\nMédia global de M = {mean_M:.5f}; t-stat = {tstat:.3f}, p-value = {pval:.3f}")

# ---------- 5. plots para aula ----------
plt.figure(figsize=(12,4))
plt.plot(df2['date'], df2['Y'], label='Y (observado)', alpha=0.6)
plt.plot(df2['date'], df2['hatY'], label=f'Previsor (média móvel k={k})', alpha=0.9)
plt.legend(); plt.title('Série e previsor'); plt.xlabel('data'); plt.tight_layout()

plt.figure(figsize=(12,3))
plt.plot(df2['date'], df2['M'])
plt.axhline(0, color='k', linestyle='--')
plt.title('Erros de previsão M_n'); plt.xlabel('data'); plt.tight_layout()

plt.figure(figsize=(12,3))
plt.plot(df2['date'], df2['S'])
plt.title('Processo acumulado S_n = sum M_i'); plt.xlabel('data'); plt.tight_layout()

plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'daily-min-temperatures.csv'

### 📌 O que os alunos devem fazer / observar

- Rodar o código e observar as figuras:
  - \(Y_n\) vs. \(\hat{Y}_n\)
  - a série de erros \(M_n\)
  - o acumulado \(S_n\)

- **Interpretação esperada**:
  - Se o previsor **não é enviesado**, a média de \(M_n\) deve ficar perto de **zero**.
  - \(S_n\) deve **oscilar sem tendência clara para cima** (não deve crescer deterministicamente).

---

### 🔍 Análise condicional

- Na tabela impressa (médias de \(M_n\) por *bins* de \(Y_{n-1}\)), verificar se as médias se aproximam de **zero**.
- Se houver viés em algum bin, discutir possíveis causas:
  - modelo de média móvel não captura sazonalidade
  - mudança de regime
  - comportamento não linear etc.

---

### 🧪 Teste t simples

- O t-test verifica a hipótese de **média global** \(E[M_n] = 0\).
- **Limitação importante**: a dependência temporal nas séries viola a suposição de independência do teste t.
- Sugestão mais robusta: **block-bootstrap**, que respeita dependência temporal.

---

### 📚 Discussão teórica curta

- Se o previsor fosse, de fato,
  \[
  \hat{Y}_n = E[Y_n \mid \mathcal{F}_{n-1}],
  \]
  então
  \[
  E[M_n \mid \mathcal{F}_{n-1}] = 0,
  \]
  e \(S_n\) seria um **martingal**.
- No exercício prático, usamos uma **estimativa empírica** da condicional (média móvel), então esperamos **aproximação** da propriedade de martingal — não exatidão.

---

### 🧠 Extensões para trabalhos de casa

- Trocar o previsor (AR(1), regressão com defasagens, Prophet etc.) e comparar erros: qual é mais “martingal-like”?
- Testar se a **variância condicional** de \(M_n\) é constante (heterocedasticidade?).
- Construir intervalos **always-valid** ou martingais exponenciais para detectar mudanças (tema avançado).

---

### 📄 Texto curto para dar aos alunos (enunciado do exercício)

Baixe o arquivo `daily-min-temperatures.csv` (URL fornecida). Construa um previsor \(\hat{Y}_n\) como média móvel de \(k=7\) dias (usando apenas informação até \(n-1\)). Defina:
\[
M_n = Y_n - \hat{Y}_n,\qquad
S_n = \sum_{t=1}^n M_t.
\]

**(a)** Plotar as séries \(Y_n\), \(\hat{Y}_n\), \(M_n\), \(S_n\).

**(b)** Agrupar os valores por quantis de \(Y_{n-1}\) e calcular a média condicional de \(M_n\). O que observa?

**(c)** Discutir por que \(S_n\) se aproxima do comportamento de um martingal se o previsor é “honesto”. Quais limitações tem a checagem empírica feita?
